# G4 systematic covariances (notebook production)

Product **B** rate covariances on final-selection (`sel_mup`) MC, matching
`syst_multisim_chunk.py` / `syst_multisim_aggregate.py`.

Per-knob packs + total from **multiplied** universe weights.

Helpers: `analysis_village.numucc_1p0pi.syst_multisim_inspect`.
Live file-walk debug remains in `systematics-multisim-live.ipynb`.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os
import sys
import time
from os import path, makedirs
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

REPO = Path('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')
sys.path.insert(0, str(REPO))

from analysis_village.numucc_1p0pi.files_config import save_fig_base_dir
from analysis_village.numucc_1p0pi.syst_disk_layout import normalized_root
from analysis_village.numucc_1p0pi.syst_multisim_common import build_var_configs
from analysis_village.numucc_1p0pi.syst_multisim_inspect import (
    cov_pack_for_knob,
    cov_pack_multiplied,
    default_syst_disk_root,
    drop_bad_weights,
    frac_unc_from_pack,
    load_and_prepare_evt,
    log,
    matrix_save_name,
    probe_weight_columns,
    syst_knob_names,
    univ_hist_save_name,
    write_multisim_npzs_and_manifest,
)
from analysis_village.numucc_1p0pi.utils import (
    dpi,
    fig_ext,
    plot_frac_unc,
    plot_heatmap,
    plot_univ_hists,
)
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig

plt.style.use('presentation.mplstyle')


In [ ]:
# --- configuration ---
SYST_NAME = 'G4'
FLUX_KNOB_GROUPS = os.environ.get('FLUX_KNOB_GROUPS', 'all')  # Flux only
VAR_SET = os.environ.get('MULTISIM_VAR_SET', 'final')  # final | intermediate | sel_all
# Optional: override with an explicit list, e.g. [VariableConfig.muon_end_x()]
var_configs = build_var_configs(VAR_SET)

cov_type = 'rate'
bkgd_subtract = True
n_univ = int(os.environ.get('MULTISIM_N_UNIV', '100'))

VERBOSE_UNIV_LOOP = True
PLOT_UNIV_EVERY_N_KNOBS = 0 if SYST_NAME == 'MCstat' else 0  # set >0 to plot every Nth knob
PLOT_FRAC_UNC_EACH_VAR = True
PLOT_HEATMAP_FIRST_VAR = True
SAVE_UNIV_HISTS = True
SAVE_FIGS = True
FIG_DPI = dpi

_tag = os.environ.get('MULTISIM_NOTEBOOK_TAG')  # default: today's date
SYST_DISK_ROOT = default_syst_disk_root(save_fig_base_dir, SYST_NAME, tag=_tag)
log(f'SYST_NAME={SYST_NAME}')
log(f'syst disk root: {normalized_root(SYST_DISK_ROOT)}')
log(f'{len(var_configs)} variables from VAR_SET={VAR_SET!r}')


## Load MC and clean weights


In [ ]:
# --- load + prepare ---
mc_evt_df = load_and_prepare_evt(SYST_NAME)
knobs = syst_knob_names(SYST_NAME, flux_groups=FLUX_KNOB_GROUPS)
probe_weight_columns(mc_evt_df, SYST_NAME, flux_groups=FLUX_KNOB_GROUPS)

log(f'dropping bad {SYST_NAME} weights ...')
t_clean = time.time()
n_before = len(mc_evt_df)
mc_evt_df = drop_bad_weights(mc_evt_df, SYST_NAME, knobs, n_univ)
log(
    f'{len(mc_evt_df):,} events after cleaning '
    f'({n_before - len(mc_evt_df):,} dropped) in {time.time() - t_clean:.1f}s'
)
if len(mc_evt_df) == 0:
    raise RuntimeError(f'no events left after weight cleaning for {SYST_NAME}')


## Compute covariances


In [ ]:
# --- compute covariances (per-knob + multiplied total) ---
syst_dict = {SYST_NAME: {}, f'{SYST_NAME}_by_knob': {}}
results_by_var = {}
t_all = time.time()
first_var_done = False

for ivar, var_config in enumerate(tqdm(var_configs, desc=f'variables ({SYST_NAME})')):
    vsn = var_config.var_save_name
    t_var = time.time()
    log(f'--- {SYST_NAME} variable [{ivar + 1}/{len(var_configs)}] {vsn} ---')

    per_knob = {}
    frac_unc_curves = []
    knob_labels = []

    for iknob, knob in enumerate(tqdm(knobs, desc=f'knobs {SYST_NAME}/{vsn}', leave=True)):
        t_knob = time.time()
        verbose_knob = VERBOSE_UNIV_LOOP and (iknob == 0 or (iknob + 1) % 10 == 0)
        try:
            pack, univ, cv = cov_pack_for_knob(
                mc_evt_df, var_config, knob, n_univ, cov_type, bkgd_subtract,
                verbose=verbose_knob,
            )
        except Exception as ex:
            log(f'  SKIP knob={knob}: {ex}')
            continue
        per_knob[knob] = pack
        fu = frac_unc_from_pack(pack)
        knob_labels.append(knob)
        frac_unc_curves.append(fu)
        log(
            f'  knob [{iknob + 1}/{len(knobs)}] {knob}: '
            f'mean frac unc={np.mean(fu):.4f}, max={np.max(fu):.4f}, '
            f'{time.time() - t_knob:.1f}s'
        )
        do_univ_plot = PLOT_UNIV_EVERY_N_KNOBS and (
            iknob == 0 or (iknob + 1) % PLOT_UNIV_EVERY_N_KNOBS == 0
        )
        if do_univ_plot:
            plot_univ_hists(
                univ, cv, ('mc', knob), var_config,
                ax_titles=['', '', ''],
                save_fig=SAVE_UNIV_HISTS, approval='',
                save_name=univ_hist_save_name(SYST_DISK_ROOT, SYST_NAME, vsn, knob) if SAVE_UNIV_HISTS else None,
            )
            plt.show()

    if not per_knob:
        log(f'  no knobs succeeded for {SYST_NAME}/{vsn}')
        continue

    log(f'  computing multiplied-weight total ({len(per_knob)} knobs) ...')
    t_tot = time.time()
    try:
        pack_total, univ_tot, cv_tot = cov_pack_multiplied(
            mc_evt_df, var_config, tuple(per_knob.keys()), n_univ, cov_type, bkgd_subtract,
            bundled_tag=SYST_NAME, verbose=VERBOSE_UNIV_LOOP,
        )
    except Exception as ex:
        log(f'  SKIP multiplied total for {SYST_NAME}/{vsn}: {ex}')
        continue

    fu_tot = frac_unc_from_pack(pack_total)
    log(
        f'  total (× weights): mean frac unc={np.mean(fu_tot):.4f}, max={np.max(fu_tot):.4f}, '
        f'{time.time() - t_tot:.1f}s'
    )
    syst_dict[SYST_NAME][vsn] = pack_total
    syst_dict[f'{SYST_NAME}_by_knob'][vsn] = per_knob
    results_by_var[vsn] = {'per_knob': per_knob, 'total_mult': pack_total}

    frac_unc_curves.append(fu_tot)
    knob_labels.append('total (weights ×)')
    if PLOT_FRAC_UNC_EACH_VAR:
        plot_frac_unc(frac_unc_curves, var_config, legends=knob_labels)
        plt.show()

    if PLOT_HEATMAP_FIRST_VAR and not first_var_done:
        plot_heatmap(
            pack_total['cov_frac'], var_config.bins,
            plot_labels=[var_config.var_labels[1], var_config.var_labels[1], 'cov_frac (total)'],
            plot=True, cmap='viridis',
        )
        plt.show()
        plot_univ_hists(
            univ_tot, cv_tot, ('mc', SYST_NAME), var_config,
            ax_titles=['', '', ''],
            save_fig=SAVE_UNIV_HISTS, approval='',
            save_name=univ_hist_save_name(SYST_DISK_ROOT, SYST_NAME, vsn, 'total-mult') if SAVE_UNIV_HISTS else None,
        )
        plt.show()
        first_var_done = True

    log(f'  done {SYST_NAME}/{vsn} in {time.time() - t_var:.1f}s')

log(f'FINISHED {SYST_NAME}: {len(syst_dict[SYST_NAME])} variables in {time.time() - t_all:.1f}s')


## Save


In [ ]:
# --- save NPZ + manifest ---
write_multisim_npzs_and_manifest(
    syst_dict, SYST_DISK_ROOT,
    syst_name=SYST_NAME,
    n_univ=n_univ,
    bkgd_subtract=bkgd_subtract,
    cov_type=cov_type,
    knobs=knobs,
)


## Inspect


In [ ]:
# --- inspect saved packs (heatmaps) ---
for vc in var_configs:
    inspect_var = vc.var_save_name
    if inspect_var not in syst_dict.get(SYST_NAME, {}):
        continue
    pack = syst_dict[SYST_NAME][inspect_var]
    print(f'Inspecting {SYST_NAME} / {inspect_var}')
    for matrix_type in ('cov', 'cov_frac', 'corr'):
        plot_heatmap(
            pack[matrix_type], vc.bins,
            plot_labels=[vc.var_labels[1], vc.var_labels[1], matrix_type],
            plot=True, cmap='viridis',
            save_fig=SAVE_FIGS,
            save_name=matrix_save_name(SYST_DISK_ROOT, SYST_NAME, inspect_var, matrix_type),
        )
        plt.show()
    print(f'{SYST_NAME} sqrt(diag(cov_frac)) for {inspect_var}:', frac_unc_from_pack(pack))
